# 10Eros v1.5 **BF16** — I2V com DMD (mesmo workflow do `_2gpu` local)

Para comparar na tela com o render **W4A8** local: use **o mesmo prompt, seed, imagem, tamanho e quadros**.
O grafo é o do `10Eros_v1.5_W4A8_I2V_DMD_2gpu.json` (2 passadas, upscaler x2 1.1, decode em tiles), trocando só o
checkpoint pelo BF16 original e o text encoder pelo Gemma heretic BF16 (a fonte do nosso W4A8).

**Precisa de A100 com RAM alta** (Runtime → Change runtime type → A100, High-RAM). O checkpoint tem 43 GB e não cabe
inteiro em 40 GB de VRAM: o ComfyUI deixa uma parte na RAM da VM — é normal, só fica mais lento.
Downloads: ~70 GB, direto do Hugging Face para a VM. Rode as células 1 → 4; depois só a 4 para cada vídeo.

Nós usados: `AudioAdjustVolume`, `CLIPTextEncode`, `CheckpointLoaderSimple`, `EmptyLTXVLatentVideo`, `GetImageSize`, `ImageResizeKJv2`, `KSamplerSelect`, `LTXAVTextEncoderLoader`, `LTXReferenceConditioning`, `LTXReferenceEnable`, `LTXVAudioVAEDecode`, `LTXVAudioVAEEncode`, `LTXVAudioVAELoader`, `LTXVConcatAVLatent`, `LTXVConditioning`, `LTXVEmptyLatentAudio`, `LTXVImgToVideoInplaceKJ`, `LTXVLatentUpsamplerTiled`, `LTXVPreprocess`, `LTXVSeparateAVLatent`, `LatentUpscaleModelLoader`, `LoadImage`, `LoraLoaderModelOnly`, `ManualSigmas`, `ResizeImageMaskNode`, `SamplerCustom`, `SetLatentNoiseMask`, `SolidMask`, `VAEDecodeTiled`, `VHS_VideoCombine`

In [ ]:
#@title 1. Instalar ComfyUI e nós (uma vez por VM)
import os, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise SystemExit(f"falhou: {c}")
os.chdir("/content")
if not os.path.isdir("/content/ComfyUI"):
    sh("git clone -q https://github.com/Comfy-Org/ComfyUI && cd ComfyUI && git checkout -q c1739380")
NOS = [("https://github.com/kijai/ComfyUI-KJNodes", "3f20054"),
       ("https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite", "4ee72c0"),
       ("https://github.com/TenStrip/10S-Comfy-nodes", "c231aca")]
for url, rev in NOS:
    d = "/content/ComfyUI/custom_nodes/" + url.rsplit("/", 1)[1]
    if not os.path.isdir(d):
        sh(f"git clone -q {url} {d} && cd {d} && git checkout -q {rev}")
# requirements do ComfyUI sem mexer no torch do Colab
def nome(l):
    for s in "=<>~![ ;":
        l = l.split(s)[0]
    return l.strip().lower()
req = [l for l in open("/content/ComfyUI/requirements.txt") if nome(l) not in ("torch", "torchvision", "torchaudio")]
open("/content/req_comfy.txt", "w").writelines(req)
sh(f"{sys.executable} -m pip install -q -r /content/req_comfy.txt")
for url, _ in NOS:
    r = "/content/ComfyUI/custom_nodes/" + url.rsplit("/", 1)[1] + "/requirements.txt"
    if os.path.isfile(r):
        sh(f"{sys.executable} -m pip install -q -r {r}")
sh("apt-get -qq install -y ffmpeg > /dev/null")
print("OK: ComfyUI e nós instalados")

In [ ]:
#@title 2. Baixar modelos do Hugging Face (~70 GB)
dmd = "LTX2.3_DMD_reshaped_r256.safetensors"  #@param ["LTX2.3_DMD_reshaped_r256.safetensors", "LTX2.3_DMD_hybrid_v2.safetensors"]
text_encoder = "gemma_3_12B_it_heretic.safetensors"  #@param ["gemma_3_12B_it_heretic.safetensors", "gemma_3_12B_it_heretic_fp8_e4m3fn.safetensors"]
import os, shutil, time
from huggingface_hub import hf_hub_download
M = "/content/ComfyUI/models"
ARQ = [("TenStrip/LTX2.3-10Eros", "10Eros_v1.5_bf16.safetensors", "checkpoints"),
       ("DreamFast/gemma-3-12b-it-heretic", "comfyui/" + text_encoder, "text_encoders"),
       ("Lightricks/LTX-2.3", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "latent_upscale_models"),
       ("TenStrip/LTX2.3_DMD_Lora", dmd, "loras")]
# um DMD e um text encoder por vez: tira os outros das pastas para a celula 4 nao escolher o errado
for pasta, prefixo, escolhido in (("loras", "LTX2.3_DMD", dmd), ("text_encoders", "gemma_3_12B_it_heretic", text_encoder)):
    for f in os.listdir(f"{M}/{pasta}"):
        if f.startswith(prefixo) and f != escolhido:
            os.makedirs("/content/fora", exist_ok=True); shutil.move(f"{M}/{pasta}/{f}", f"/content/fora/{f}")
            print("guardado fora:", f)
    if os.path.isfile(f"/content/fora/{escolhido}"):
        shutil.move(f"/content/fora/{escolhido}", f"{M}/{pasta}/{escolhido}")
for repo, arq, pasta in ARQ:
    dest = f"{M}/{pasta}/{os.path.basename(arq)}"
    if os.path.isfile(dest):
        print("ja existe", dest); continue
    t0 = time.time()
    p = hf_hub_download(repo, arq, local_dir="/content/hf")
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(p, dest)
    print(f"{dest}  {os.path.getsize(dest)/2**30:.1f} GiB em {time.time()-t0:.0f} s")
print("OK: modelos prontos")

In [ ]:
#@title 3. Subir o ComfyUI em segundo plano
import subprocess, time, urllib.request, sys
LOG = open("/content/comfy.log", "a")
try:
    urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=2); print("ja estava no ar")
except Exception:
    # start_new_session: cancelar uma celula (Ctrl+M I / botao parar) NAO derruba o ComfyUI junto
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd="/content/ComfyUI",
                     stdout=LOG, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=2); break
        except Exception:
            time.sleep(5)
    else:
        print(open("/content/comfy.log").read()[-3000:]); raise SystemExit("ComfyUI nao subiu")
    print("ComfyUI no ar")
print(subprocess.run("grep -iE 'IMPORT FAILED|Cannot import' /content/comfy.log | tail -5", shell=True,
                     capture_output=True, text=True).stdout or "sem erro de import de nos")

In [ ]:
#@title 4. Gerar (use os MESMOS valores do render W4A8 local)
prompt = "A woman looks at the camera and says: \"Hello, this is a test of the audio.\" Natural lighting, subtle movement."  #@param {type:"string"}
negativo = "3D, phasing, captions, VR, still image, bad quality, subtitles, text, watermark, overlay effects, pc game, yelling, console game, video game, cartoon, childish, ugly, text, blur, logo, wordmark, static, low quality, noise, white noise, bleep, censoring, censor, bleeping, beep, beeping, newscast, interview, podcast, non-english, foreign language, russian, chinese, japanese, mutant, horror, 70's, film grain, cinematic, comedy, stand-up "  #@param {type:"string"}
seed = 635141064074927  #@param {type:"integer"}
largura = 1024  #@param {type:"integer"}
altura = 1376  #@param {type:"integer"}
quadros = 360  #@param {type:"integer"}
fps = 24  #@param {type:"number"}
forca_dmd = 1.0  #@param {type:"number"}
enviar_imagem = True  #@param {type:"boolean"}
import json, os, time, glob, shutil, base64, urllib.request, urllib.error
from IPython.display import HTML, display
if enviar_imagem:
    from google.colab import files
    up = files.upload()
    nome_img = list(up)[0]
    shutil.move(nome_img, f"/content/ComfyUI/input/{nome_img}")
else:
    nome_img = max(os.listdir("/content/ComfyUI/input"), key=lambda f: os.path.getmtime(f"/content/ComfyUI/input/{f}"))
lora = [f for f in os.listdir("/content/ComfyUI/models/loras") if f.startswith("LTX2.3_DMD")][0]
te = [f for f in os.listdir("/content/ComfyUI/models/text_encoders") if f.startswith("gemma_3_12B_it_heretic")][0]
print("imagem:", nome_img, "| DMD:", lora, "| text encoder:", te)
TEMPLATE = json.loads('{"274": {"inputs": {"audio": ["550", 0], "audio_vae": ["617", 0]}, "class_type": "LTXVAudioVAEEncode", "_meta": {"title": "LTXV Audio VAE Encode"}}, "275": {"inputs": {"samples": ["274", 0], "mask": ["630", 0]}, "class_type": "SetLatentNoiseMask", "_meta": {"title": "Set Latent Noise Mask"}}, "523": {"inputs": {"frame_rate": "@FPS@", "positive": ["536", 0], "negative": ["537", 0]}, "class_type": "LTXVConditioning", "_meta": {"title": "LTXVConditioning"}}, "534": {"inputs": {"width": ["893", 1], "height": ["893", 2], "length": "@QUADROS@", "batch_size": 1}, "class_type": "EmptyLTXVLatentVideo", "_meta": {"title": "EmptyLTXVLatentVideo"}}, "535": {"inputs": {"frames_number": "@QUADROS@", "frame_rate": "@FPS_INT@", "batch_size": 1, "audio_vae": ["617", 0]}, "class_type": "LTXVEmptyLatentAudio", "_meta": {"title": "LTXV Empty Latent Audio"}}, "536": {"inputs": {"text": "@PROMPT@", "tokens": "260 / 77 tokens (≈)", "clip": ["616", 0]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP Text Encode (Prompt)"}}, "537": {"inputs": {"text": "@NEGATIVO@", "tokens": "74 / 77 tokens (≈)", "clip": ["616", 0]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP Text Encode (Prompt)"}}, "548": {"inputs": {"video_latent": ["772", 0], "audio_latent": ["535", 0]}, "class_type": "LTXVConcatAVLatent", "_meta": {"title": "LTXVConcatAVLatent"}}, "549": {"inputs": {"frame_rate": "@FPS@", "loop_count": 0, "filename_prefix": "Eros/BF16_firstpass", "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 19, "save_metadata": true, "trim_to_audio": false, "pingpong": false, "save_output": true, "images": ["552", 0], "audio": ["551", 0]}, "class_type": "VHS_VideoCombine", "_meta": {"title": "Video Combine 🎥🅥🅗🅢"}}, "550": {"inputs": {"samples": ["556", 1], "audio_vae": ["617", 0]}, "class_type": "LTXVAudioVAEDecode", "_meta": {"title": "LTXV Audio VAE Decode"}}, "551": {"inputs": {"volume": 0, "audio": ["550", 0]}, "class_type": "AudioAdjustVolume", "_meta": {"title": "Adjust Audio Volume"}}, "552": {"class_type": "VAEDecodeTiled", "inputs": {"samples": ["556", 0], "vae": ["646", 2], "tile_size": 512, "overlap": 64, "temporal_size": 64, "temporal_overlap": 8}}, "556": {"inputs": {"av_latent": ["888", 1]}, "class_type": "LTXVSeparateAVLatent", "_meta": {"title": "LTXVSeparateAVLatent"}}, "582": {"inputs": {"sigmas": "0.92, 0.725, 0.421875, 0.0"}, "class_type": "ManualSigmas", "_meta": {"title": "Sigmas Upscale"}}, "591": {"inputs": {"video_latent": ["770", 0], "audio_latent": ["556", 1]}, "class_type": "LTXVConcatAVLatent", "_meta": {"title": "LTXVConcatAVLatent"}}, "593": {"inputs": {"samples": ["596", 1], "audio_vae": ["617", 0]}, "class_type": "LTXVAudioVAEDecode", "_meta": {"title": "LTXV Audio VAE Decode"}}, "596": {"inputs": {"av_latent": ["891", 1]}, "class_type": "LTXVSeparateAVLatent", "_meta": {"title": "LTXVSeparateAVLatent"}}, "597": {"inputs": {"frame_rate": "@FPS@", "loop_count": 0, "filename_prefix": "Eros/BF16", "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 24, "save_metadata": true, "trim_to_audio": false, "pingpong": false, "save_output": true, "images": ["740", 0], "audio": ["598", 0]}, "class_type": "VHS_VideoCombine", "_meta": {"title": "Video Combine 🎥🅥🅗🅢"}}, "598": {"inputs": {"volume": 0, "audio": ["593", 0]}, "class_type": "AudioAdjustVolume", "_meta": {"title": "Adjust Audio Volume"}}, "616": {"class_type": "LTXAVTextEncoderLoader", "inputs": {"text_encoder": "@TE@", "ckpt_name": "10Eros_v1.5_bf16.safetensors", "device": "default"}}, "617": {"inputs": {"ckpt_name": "10Eros_v1.5_bf16.safetensors"}, "class_type": "LTXVAudioVAELoader", "_meta": {"title": "Audio VAE (componente, identico ao do 10Eros)"}}, "630": {"inputs": {"value": 0, "width": 1024, "height": 1024}, "class_type": "SolidMask", "_meta": {"title": "Create Solid Mask"}}, "646": {"inputs": {"ckpt_name": "10Eros_v1.5_bf16.safetensors"}, "class_type": "CheckpointLoaderSimple", "_meta": {"title": "Load Checkpoint"}}, "740": {"inputs": {"tile_size": 512, "overlap": 64, "temporal_size": 64, "temporal_overlap": 8, "samples": ["596", 0], "vae": ["646", 2]}, "class_type": "VAEDecodeTiled", "_meta": {"title": "VAE Decode (Tiled) 2a passada"}}, "743": {"inputs": {"model_name": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors"}, "class_type": "LatentUpscaleModelLoader", "_meta": {"title": "Load Latent Upscale Model"}}, "744": {"inputs": {"tile_size": 11, "overlap": 6, "max_size_for_no_tile": 22, "rotate_for_landscape": false, "debug": false, "samples": ["556", 0], "upscale_model": ["743", 0], "vae": ["646", 2]}, "class_type": "LTXVLatentUpsamplerTiled", "_meta": {"title": "🔍 LTX Latent Upsampler (Tiled)"}}, "770": {"inputs": {"num_images": "1", "num_images.strength_1": 1.0, "num_images.index_1": 0, "vae": ["646", 2], "latent": ["744", 0], "num_images.image_1": ["905", 0]}, "class_type": "LTXVImgToVideoInplaceKJ", "_meta": {"title": "LTXVImgToVideoInplaceKJ"}}, "772": {"inputs": {"num_images": "1", "num_images.strength_1": 1.0, "num_images.index_1": 0, "vae": ["646", 2], "latent": ["534", 0], "num_images.image_1": ["900", 0]}, "class_type": "LTXVImgToVideoInplaceKJ", "_meta": {"title": "LTXVImgToVideoInplaceKJ"}}, "837": {"inputs": {"image": "@IMAGEM@"}, "class_type": "LoadImage", "_meta": {"title": "Load Image"}, "is_changed": ["da5ef5dc8a89d4adc3c2c6c6a70e013eb92f197a8f3dc9be515a7ea0eeb209dd"]}, "860": {"inputs": {"strength": 1.0, "position_mode": "reference", "verbose": false, "model": ["906", 0], "vae": ["646", 2], "image": ["907", 0], "target_latent": ["772", 0]}, "class_type": "LTXReferenceConditioning", "_meta": {"title": "🎴 LTX Reference Conditioning"}}, "868": {"inputs": {"zero_ref_timesteps": false, "verbose": false, "model": ["646", 0]}, "class_type": "LTXReferenceEnable", "_meta": {"title": "🔗 LTX Reference Enable"}}, "870": {"inputs": {"strength": 1.0, "position_mode": "reference", "verbose": false, "model": ["906", 0], "vae": ["646", 2], "image": ["905", 0], "target_latent": ["770", 0]}, "class_type": "LTXReferenceConditioning", "_meta": {"title": "🎴 LTX Reference Conditioning"}}, "871": {"inputs": {"width": "@LARGURA@", "height": "@ALTURA@", "upscale_method": "area", "keep_proportion": "crop", "pad_color": "0, 0, 0", "crop_position": "center", "divisible_by": 32, "device": "cpu", "image": ["837", 0]}, "class_type": "ImageResizeKJv2", "_meta": {"title": "Resize Image v2"}}, "872": {"inputs": {"resize_type": "scale by multiplier", "resize_type.multiplier": 0.5, "scale_method": "area", "input": ["871", 0]}, "class_type": "ResizeImageMaskNode", "_meta": {"title": "Resize Image/Mask"}}, "888": {"inputs": {"add_noise": true, "noise_seed": "@SEED@", "cfg": 1.0, "model": ["860", 0], "positive": ["523", 0], "negative": ["523", 1], "sampler": ["910", 0], "sigmas": ["914", 0], "latent_image": ["548", 0]}, "class_type": "SamplerCustom", "_meta": {"title": "First Pass"}}, "891": {"inputs": {"add_noise": true, "noise_seed": "@SEED@", "cfg": 1.0, "model": ["870", 0], "positive": ["523", 0], "negative": ["523", 1], "sampler": ["911", 0], "sigmas": ["582", 0], "latent_image": ["591", 0]}, "class_type": "SamplerCustom", "_meta": {"title": "Upscale Pass"}}, "893": {"inputs": {"width": 0, "height": 0, "upscale_method": "area", "keep_proportion": "crop", "pad_color": "0, 0, 0", "crop_position": "center", "divisible_by": 32, "device": "cpu", "image": ["872", 0]}, "class_type": "ImageResizeKJv2", "_meta": {"title": "Resize Image v2"}}, "896": {"inputs": {"resize_type": "scale by multiplier", "resize_type.multiplier": 2.0, "scale_method": "lanczos", "input": ["893", 0]}, "class_type": "ResizeImageMaskNode", "_meta": {"title": "Resize Image/Mask"}}, "898": {"inputs": {"width": ["899", 0], "height": ["899", 1], "upscale_method": "area", "keep_proportion": "crop", "pad_color": "0, 0, 0", "crop_position": "center", "divisible_by": 32, "device": "cpu", "image": ["837", 0]}, "class_type": "ImageResizeKJv2", "_meta": {"title": "Resize Image v2"}}, "899": {"inputs": {"image": ["896", 0]}, "class_type": "GetImageSize", "_meta": {"title": "Get Image Size"}}, "900": {"inputs": {"img_compression": 35, "image": ["893", 0]}, "class_type": "LTXVPreprocess", "_meta": {"title": "LTXV Preprocess"}}, "905": {"inputs": {"img_compression": 30, "image": ["898", 0]}, "class_type": "LTXVPreprocess", "_meta": {"title": "LTXV Preprocess"}}, "906": {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "@LORA@", "strength_model": "@LORA_FORCA@", "model": ["868", 0]}}, "907": {"inputs": {"img_compression": 35, "image": ["893", 0]}, "class_type": "LTXVPreprocess", "_meta": {"title": "LTXV Preprocess"}}, "910": {"inputs": {"sampler_name": "euler_ancestral"}, "class_type": "KSamplerSelect", "_meta": {"title": "KSamplerSelect"}}, "911": {"inputs": {"sampler_name": "euler_ancestral_cfg_pp"}, "class_type": "KSamplerSelect", "_meta": {"title": "KSamplerSelect"}}, "914": {"inputs": {"sigmas": "1.000, 0.955, 0.893, 0.812, 0.715, 0.603, 0.482, 0.241, 0.121, 0.0"}, "class_type": "ManualSigmas", "_meta": {"title": "Sigmas First Pass"}}}')
VAL = {"@SEED@": int(seed), "@FPS@": float(fps), "@FPS_INT@": int(round(fps)), "@LARGURA@": int(largura),
       "@ALTURA@": int(altura), "@QUADROS@": int(quadros) + 1, "@LORA@": lora, "@LORA_FORCA@": float(forca_dmd),
       "@TE@": te, "@IMAGEM@": nome_img, "@PROMPT@": prompt, "@NEGATIVO@": negativo}
def preenche(x):
    if isinstance(x, dict): return {k: preenche(v) for k, v in x.items()}
    if isinstance(x, list): return [preenche(v) for v in x]
    return VAL.get(x, x) if isinstance(x, str) else x
grafo = preenche(TEMPLATE)
antes = set(glob.glob("/content/ComfyUI/output/Eros/BF16_*.mp4"))
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps({"prompt": grafo}).encode(),
                             headers={"Content-Type": "application/json"})
try:
    pid = json.loads(urllib.request.urlopen(req).read())["prompt_id"]
except urllib.error.HTTPError as e:
    print(e.read().decode()[:3000]); raise
print("enviado", pid, "- acompanhe abaixo (a 1a vez le 43 GB do disco, demora)")
t0, visto = time.time(), len(open("/content/comfy.log", errors="replace").read())
def cancela():
    # parar a celula so para de acompanhar; isto cancela a geracao no ComfyUI sem matar o servidor
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/interrupt", data=b"", method="POST"))
    print("geracao cancelada no ComfyUI (servidor continua no ar)")
try:
  while True:
      time.sleep(15)
      log = open("/content/comfy.log", errors="replace").read()
      novo = log[visto:]; visto = len(log)
      for l in novo.splitlines():
          if any(s in l for s in ("loaded", "Requested", "Prompt executed", "Error", "Traceback", "out of memory")):
              print(f"[{time.time()-t0:5.0f}s] {l[:200]}")
      h = json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}").read())
      if pid in h:
          st = h[pid]["status"]; print("status:", st.get("status_str"), f"em {time.time()-t0:.0f} s")
          if st.get("status_str") != "success": print(json.dumps(st.get("messages", []))[-3000:])
          break
except KeyboardInterrupt:
    cancela(); raise
novos = sorted(set(glob.glob("/content/ComfyUI/output/Eros/BF16_*.mp4")) - antes)
final = [f for f in novos if "firstpass" not in f and not f.endswith("-audio.mp4")] or novos
for f in final:
    print(f)
    b64 = base64.b64encode(open(f, "rb").read()).decode()
    display(HTML(f'<video controls width="512" src="data:video/mp4;base64,{b64}"></video>'))
print("Para baixar: from google.colab import files; files.download(caminho) — ou o painel de arquivos à esquerda.")